In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset, IterableDataset
import sys
print(torch.cuda.is_available())
sys.path.append("/mnt/home/lserrano/disco-ball/")

import numpy as np
import random
import matplotlib.pyplot as plt
import h5py
import os
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR # A more standard scheduler
from einops import rearrange # A more standard import
from tqdm import tqdm
import torch.nn.functional as F

In [ ]:
from models import DISCOHouse, vectors_to_parameters
from src.advection_diffusion import Fractaloid, FractaloidPhase
from train.train import DISCOLitModule, advection_diffusion_analytical
from src.plot_dataset_samples import plot_prediction_vs_ground_truth

In [ ]:
class RelativeL2(nn.Module):
    def forward(self, x, y, aggregate="mean"):
        x = rearrange(x, "b ... -> b (...)")
        y = rearrange(y, "b ... -> b (...)")
        diff_norms = torch.linalg.norm(x - y, ord=2, dim=-1)
        y_norms = torch.linalg.norm(y, ord=2, dim=-1)

        if aggregate == "mean":
            return (diff_norms / y_norms).mean()
        else:
            return (diff_norms / y_norms)

In [ ]:
def autoregressive_predict(model, initial_seq, n_pred, device):
    preds = []
    current = initial_seq.clone().to(device)
    n_input = current.shape[1]
    for t in range(n_pred):
        inp = current[:, -n_input:].to(device)
        with torch.no_grad():
            state_labels = torch.tensor([0], device=inp.device)
            next_frame, metadata = model(inp, state_labels, n_future_steps=1)
            if t == 0:
                theta = metadata['theta_latent']
        current = torch.cat([current, next_frame], axis=1)
        preds.append(next_frame)
    return torch.cat(preds, axis=1), theta

In [ ]:
class TemporalBatchDatasetFly(IterableDataset):
    def __init__(self, n_batches, batch_size, sub_x, sub_t, split="train", input_frames=16, output_frames=2,
                 L=16.0, nx=256, nt=100, T=10.0,
                 v_range=(0.01, 1.0), D_range=(0.01, 1.0),
                 fractal_degree=8, fractal_power=2, seed=None):
        self.n_batches = n_batches
        self.batch_size = batch_size
        self.sub_x = sub_x
        self.sub_t = sub_t
        self.split = split
        self.input_frames = input_frames
        self.output_frames = output_frames
        self.L = L
        self.nx = nx
        self.nt = nt
        self.T = T
        self.v_range = v_range
        self.D_range = D_range
        self.fractal_degree = fractal_degree
        self.fractal_power = fractal_power
        self.seed = seed
        self.rng = np.random.default_rng(seed)

    def __iter__(self):
        for _ in range(self.n_batches):
            input_frames = self.input_frames
            batch_inputs = []
            batch_targets = []
            batch_v = []
            batch_d = []
            batch_init = []
            for _ in range(self.batch_size):
                # Sample advection speed and viscosity
                if self.split == 'train':
                    if random.random() < 0.5:
                        v = self.rng.uniform(*self.v_range) if isinstance(self.v_range, (tuple, list)) else float(self.v_range)
                        D = 0
                    else:
                        v = 0
                        D = self.rng.uniform(*self.D_range) if isinstance(self.D_range, (tuple, list)) else float(self.D_range)
                else:
                    v = self.rng.uniform(*self.v_range) if isinstance(self.v_range, (tuple, list)) else float(self.v_range)
                    D = self.rng.uniform(*self.D_range) if isinstance(self.D_range, (tuple, list)) else float(self.D_range)
                # Generate fractaloid initial condition
                fractaloid = FractaloidPhase(
                    degree=self.fractal_degree,
                    power=self.fractal_power,
                    size=self.nx,
                    patch_size=self.nx
                )
                u0 = fractaloid.generate(batch_size=1, seed=None).squeeze(0).numpy()
                u0 = (u0 - u0.mean()) / (u0.std() + 1e-8)
                u_xt, x, t = advection_diffusion_analytical(
                    u0, L=self.L, v=v, D=D, nt=self.nt, T=self.T
                )
                u_xt = u_xt[::self.sub_t, ::self.sub_x]
                max_start_index_input = u_xt.shape[0] - input_frames
                input = u_xt[:input_frames].copy()
                target = u_xt[input_frames: input_frames + self.output_frames].copy()
                batch_inputs.append(torch.from_numpy(input).unsqueeze(-2).float())
                batch_targets.append(torch.from_numpy(target).unsqueeze(-2).float())
                batch_v.append(v)
                batch_d.append(D)
                batch_init.append(torch.from_numpy(u0))
            batch = {
                'input': torch.stack(batch_inputs),
                'target': torch.stack(batch_targets),
                'velocities': batch_v,
                'diffusivities': batch_d,
                'initial_conditions': torch.stack(batch_init)
            }
            yield batch

In [ ]:
batch_size=4
sub_x=1
sub_t=1
n_input_frames=16
n_output_frames=50-16 #50-n_input_frames
relative_l2_error = RelativeL2()

In [ ]:
n_batches = int(1000//batch_size)  # or set as needed for your epoch size
split="train"
train_ds = TemporalBatchDatasetFly(
    n_batches=n_batches,
    batch_size=batch_size,
    sub_x=sub_x,
    sub_t=sub_t,
    split=split,
    input_frames=n_input_frames,
    output_frames=n_output_frames,
    L=16.0,
    nx=256,
    nt=100,
    T=10.0,
    fractal_power=3.0,
    fractal_degree=256, # nx
    v_range=(0.9, 1.0),#(0.9,1),#(0.01, 1.0),
    D_range=(0.9, 1.0),#(0.001, 1.0),0.9, 0.91
)
train_loader = DataLoader(train_ds, batch_size=None, num_workers=4, prefetch_factor=4, pin_memory=True)

In [ ]:
#theta_path = "/mnt/home/lserrano/disco-ball/results/advection_diffusion/dense"
device="cuda" if torch.cuda.is_available() else "cpu"
#ckpt_time="DISCO_advection-diffusion_solverrk4_adjFalse_h128_t2_steps1_initFalse_bs64_lr0.0005_ctxTrue_noise0.0001_inframes16_outframes2_T10"
ckpt_time="DISCO_advection-diffusion_solverrk4_adjFalse_h128_t2_steps1_initFalse_bs64_lr0.0005_ctxTrue_noise0.0001_inframes16_outframes2_T10"
ckpt_path = f"/mnt/home/lserrano/disco-ball/outputs/{ckpt_time}/last-v1.ckpt" #model_final.ckpt"
print(f"Loading model from {ckpt_path}...")
model = DISCOLitModule.load_from_checkpoint(ckpt_path, map_location=device)
model = model.model.to(device)
model.eval()

# I. Finetune theta

In [ ]:
results_dir = f"results/{ckpt_time}"
dataset_name="advection_diffusion"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(f"{results_dir}/{dataset_name}/plots/", exist_ok=True)
os.makedirs(f"{results_dir}/{dataset_name}/predictions/", exist_ok=True)
os.makedirs(f"{results_dir}/{dataset_name}/theta/", exist_ok=True)
os.makedirs(f"{results_dir}/{dataset_name}/errors/", exist_ok=True)

In [ ]:
n=0
rollout_error=0
all_theta=[]
all_velocities = []
all_diffusivities = []

for batch in tqdm(train_loader):
    inp, target = batch["input"], batch["target"]
    all_velocities += batch["velocities"]
    all_diffusivities += batch["diffusivities"]
    break

In [ ]:
all_diffusivities
all_velocities

In [ ]:
inp = inp.to(device)
target = target.to(device)
state_labels = torch.tensor([0], device=inp.device)
n_future_steps=34

x_shape = inp.shape
B, T, C = x_shape[:3]
spatial = x_shape[3:]
dim = len(spatial)

n_sample = inp.shape[0]
#pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)

# encode into 2 dimensional
theta_latent, metadata= model.encode_theta_latent(inp, state_labels)

# decode into 100k parameters
theta = model.decode_theta(theta_latent, dim)

#predict
with torch.no_grad():
    pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=n_future_steps, integration_time=n_future_steps, predict_normed=False, metadata=metadata)
rollout_error = relative_l2_error(pred, target).item()

print(f"Initial error", rollout_error)

In [ ]:
inp_train = inp.clone()
target_train = target.clone()
trajectory_train = torch.cat([inp_train, target_train], axis=1)
alpha_train = torch.eye(batch_size)

In [ ]:
def get_operators(theta_list, model, dim=1):
    """
    Create operator neural networks that can be called with (x, state_labels).
    """
    from torch.func import functional_call  # Make sure to import this
    
    base_opnn = model.opnns[str(dim)]
    param_dict = dict(base_opnn.named_parameters())
    
    operators = []
    for theta in theta_list:
        # Convert theta vector to parameter dictionary
        batched_params_dict = vectors_to_parameters(theta.unsqueeze(0), param_dict)
        params_for_this_theta = {k: v[0] for k, v in batched_params_dict.items()}
        
        # Create operator that can be called with (x, state_labels)
        def make_operator(params):
            def operator(x, state_labels):
                return functional_call(base_opnn, params, (x, state_labels))
            return operator
        
        operators.append(make_operator(params_for_this_theta))
    
    return operators

def get_batched_operators(theta_batch, model, dim=1):
    """
    Create a single function that processes all thetas at once.
    """
    from torch.func import functional_call, vmap
    
    base_opnn = model.opnns[str(dim)]
    param_dict = dict(base_opnn.named_parameters())
    batched_params_dict = vectors_to_parameters(theta_batch, param_dict)

    
    
    def batched_operator(x_batch, state_labels_batch):
        print('x_batch', x_batch.shape)
        print('theta_batch', theta_batch.shape)
        print('state_labels_batch', state_labels_batch.shape)
        return vmap(functional_call, in_dims=(None, 0, 0))(
            base_opnn, batched_params_dict, (x_batch, state_labels_batch)
        )
    
    return batched_operator

In [ ]:
def get_batched_operators(theta_batch, model, dim=1):
    """
    Create a single function that processes all thetas at once.
    theta_batch: [num_operators, theta_dim]
    """
    from torch.func import functional_call, vmap
    
    base_opnn = model.opnns[str(dim)]
    param_dict = dict(base_opnn.named_parameters())
    batched_params_dict = vectors_to_parameters(theta_batch, param_dict)
    
    def batched_operator(x, state_labels):
        # x: [batch_size, channels, ...spatial]
        # state_labels: [num_states] or [batch_size, num_states]
        # theta_batch: [num_operators, theta_dim]
        
        num_operators = theta_batch.shape[0]
        
        # We need to replicate x and state_labels for each operator
        # x needs to become [num_operators, batch_size, channels, ...spatial]
        x_replicated = x.unsqueeze(0).expand(num_operators, -1, -1, -1)
        
        # state_labels needs to become [num_operators, ...]
        if state_labels.dim() == 1:
            # [num_states] -> [num_operators, num_states]
            state_labels_replicated = state_labels.unsqueeze(0).expand(num_operators, -1)
        else:
            # [batch_size, num_states] -> [num_operators, batch_size, num_states]  
            state_labels_replicated = state_labels.unsqueeze(0).expand(num_operators, -1, -1)
        
        # Now vmap over the operator dimension (first dim)
        return vmap(functional_call, in_dims=(None, 0, 0))(
            base_opnn, batched_params_dict, (x_replicated, state_labels_replicated)
        )
    
    return batched_operator

In [ ]:
# Get your operators
#x = inp[:,-1].cuda()
#with torch.no_grad():
#    operators = get_operators(theta, model, dim=1)

# Use them - each operator can be called with (x, state_labels)
#with torch.no_grad():
#    for i, operator in enumerate(operators):
#        result = operator(x, state_labels)
#        print(f"Operator {i} output shape: {result.shape}")

In [ ]:
x = inp[:,-1].cuda()
with torch.no_grad():
    operators = get_batched_operators(theta, model, dim=1)
with torch.no_grad():
    result = operators(x, state_labels)
    print(f"Operators output shape: {result.shape}")
num_operators=result.shape[0]

In [ ]:
batch_size=1
n_batches = int(1000//batch_size)  # or set as needed for your epoch size
split="test"
test_ds = TemporalBatchDatasetFly(
    n_batches=n_batches,
    batch_size=batch_size,
    sub_x=sub_x,
    sub_t=sub_t,
    split=split,
    input_frames=n_input_frames,
    output_frames=n_output_frames,
    L=16.0,
    nx=256,
    nt=100,
    T=10.0,
    fractal_power=3.0,
    fractal_degree=256, # nx
    v_range=(2.0, 2.01),#(1,1.01),#(1, 1.01),#(1, 1.1),#(0.01, 1.0),
    D_range=(0., 0.000001) #(0.9, 0.91),#(0.9, 0.91)#(0.9, 0.91),#(0.9, 0.91), #(5, 5.001),#(0.9, 0.91),#(5, 5.001),#(0.001, 1.0),
)
test_loader = DataLoader(test_ds, batch_size=None, num_workers=4, prefetch_factor=4, pin_memory=True)

In [ ]:
for batch in tqdm(test_loader):
    inp, target = batch["input"], batch["target"]
    all_velocities += batch["velocities"]
    all_diffusivities += batch["diffusivities"]
    break

In [ ]:
for idx in range(inp.shape[0]):
    plt.plot(inp[idx, 0].squeeze())

In [ ]:
idx=0
for t in range(inp.shape[1]):
    plt.plot(inp[idx, t].squeeze())

In [ ]:
def mixture_of_operators(x, state_labels, alpha, operators):
    """
    Compute weighted mixture of operators using einsum.
    """
    # Get operator outputs: [num_operators, batch_size, channels, ...spatial]
    operator_outputs = operators(x, state_labels)
    
    if alpha.dim() == 1:
        # alpha: [num_operators]
        # Use einsum to weight and sum over operators
        return torch.einsum('k,k...->...', alpha, operator_outputs)
    else:
        # alpha: [batch_size, num_operators]
        # Transpose to [num_operators, batch_size] and use einsum
        alpha_t = alpha.t()  # [num_operators, batch_size]
        return torch.einsum('kb,kb...->b...', alpha_t, operator_outputs)

In [ ]:
def solve_mixture_ode(x_input, operators, alpha, state_labels, 
                     integration_time=1.0, n_future_steps=1, 
                     solver='rk4', rtol=1e-7):
    """
    Solve neural ODE using mixture of operators.
    """
    from torchdiffeq import odeint
    
    # Create ODE function
    def ode_func(t, x):
        return mixture_of_operators(x, state_labels, alpha, operators)
    
    # Time grid
    t = torch.linspace(0, integration_time, n_future_steps + 1, device=x_input.device)
    
    # Solve ODE
    nsteps, solution = odeint(ode_func, x_input, t=t, rtol=rtol, method=solver)
    
    # Return future steps (excluding initial condition)
    return solution[1:, ...]

In [ ]:
# --- Hyperparameters and setup ---
num_steps = 1
epochs = 500

# Test data for extrapolation
x_test = inp[:, -1].to(device)
y_test = target.to(device)

training_horizon = 1
n_future_steps = 34

# --- Optimizer and Scheduler ---
alpha = torch.zeros(x_test.shape[0], num_operators).cuda().clone().detach().requires_grad_()
lambd = 1e-2
optimizer = torch.optim.AdamW([alpha], lr=1e-1, weight_decay=0)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
num_training_frames = 1

print("Starting training...")

for epoch in tqdm(range(epochs), desc="Training"):
    
    # Random time step for training
    t = random.randint(0, n_input_frames - training_horizon - 1)
    x_train = inp[:, t].to(device)
    y_train = inp[:, t+1:t+1+training_horizon].to(device)
    y_train = rearrange(y_train, "b t c h -> t b c h")

    with torch.no_grad():
        operators = get_batched_operators(theta, model, dim=1)

    pred = solve_mixture_ode(
    x_train, operators, alpha, state_labels,
    integration_time=training_horizon, n_future_steps=training_horizon)
    
    # Calculate training loss
    rel_loss = relative_l2_error(pred, y_train)
    sparsity_loss = alpha.abs().sum(-1).mean()
    loss = rel_loss + lambd*sparsity_loss
    
    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    
    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Train Loss: {rel_loss.item():.6f} | "
        f"Sparsity Loss: {sparsity_loss.item():.6f} | "
        #f"Test Error: {test_error:.6f}"
    )

print("Training finished.")

In [ ]:
with torch.no_grad():
    pred = solve_mixture_ode(
        x_test, operators, alpha, state_labels,
        integration_time=n_output_frames, n_future_steps=n_output_frames)

In [ ]:
yhat = pred.detach().cpu()
y = rearrange(target.clone(), "b t c h -> t b c h")

In [ ]:
relative_l2_error(yhat, y)

In [ ]:
alpha[0]

In [ ]:
idx=0
for t in range(yhat.shape[0]):
    plt.plot(yhat[t, idx].squeeze())

In [ ]:
for t in range(yhat.shape[0]):
    plt.plot(y[t, idx].squeeze())

In [ ]:
# --- Hyperparameters and setup ---
num_steps = 1
epochs = 500 #500

# Test data for extrapolation
x_test = inp[:, -1].to(device)
y_test = target.to(device)

#num_operators = 16
training_horizon = 1
n_future_steps = 34
noise_level=1e-3

# --- Optimizer and Scheduler ---
#alpha = (0.*torch.randn(x_test.shape[0], num_operators)).cuda().clone().detach().requires_grad_()

theta_ = theta.clone().detach().requires_grad_()
lambd = 1e-2
optimizer = torch.optim.AdamW([
    {'params': [alpha], 'lr': 1e-2},          # Learning rate for alpha
    {'params': [theta_], 'lr': 1e-4}      # Learning rate for theta
], weight_decay=0)

scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
num_training_frames = 1

print("Starting training...")

for epoch in tqdm(range(epochs), desc="Training"):
    
    # Random time step for new trajectory
    t = random.randint(0, n_input_frames - training_horizon - 1)
    x_train = inp[:, t].to(device)
    x_train = x_train+ torch.randn_like(x_train)*noise_level
    y_train = inp[:, t+1:t+1+training_horizon].to(device)
    y_train = rearrange(y_train, "b t c h -> t b c h")

    # Random time step for old trajectory

    t = random.randint(0, n_input_frames+n_output_frames - training_horizon - 1)
    x_train_ = trajectory_train[:, t].to(device)
    x_train_ = x_train_ + torch.randn_like(x_train)*noise_level
    y_train_ = trajectory_train[:, t+1:t+1+training_horizon].to(device)
    y_train_ = rearrange(y_train_, "b t c h -> t b c h")

    operators = get_batched_operators(theta_, model, dim=1)

    pred = solve_mixture_ode(
    x_train, operators, alpha, state_labels,
    integration_time=training_horizon, n_future_steps=training_horizon)

    pred_ = solve_mixture_ode(
    x_train_, operators, alpha_train.cuda(), state_labels,
    integration_time=training_horizon, n_future_steps=training_horizon)
    
    # Calculate training loss
    rel_loss = relative_l2_error(pred, y_train)
    rel_loss_2 = relative_l2_error(pred_, y_train_)
    sparsity_loss = alpha.abs().sum(-1).mean()
    loss = rel_loss + rel_loss_2 + lambd*sparsity_loss
    
    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    
    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Train Loss: {rel_loss.item():.6f} | "
        f"Train Loss 2: {rel_loss_2.item():.6f} | "
        f"Sparsity Loss: {sparsity_loss.item():.6f} | "
        #f"Test Error: {test_error:.6f}"
    )

print("Training finished.")

In [ ]:
with torch.no_grad():
    pred = solve_mixture_ode(
        x_test, operators, alpha, state_labels,
        integration_time=n_output_frames, n_future_steps=n_output_frames)

In [ ]:
yhat = pred.detach().cpu()
y = rearrange(target.clone(), "b t c h -> t b c h")

In [ ]:
relative_l2_error(rearrange(yhat, "t b c h -> b t c h"), rearrange(y, "t b c h -> b t c h"))

In [ ]:
y.shape

In [ ]:
idx=0
for t in range(10):
    plt.plot(yhat[t, idx].squeeze())

In [ ]:
for t in range(30):
    plt.plot(y[t, idx].squeeze())

In [ ]:
plt.bar(range(alpha.shape[1]), alpha.cpu().detach()[0])
plt.title('Alpha Values')
plt.xlabel('Index')
plt.ylabel('Alpha')
plt.show()

In [ ]:
plt.bar(range(alpha.shape[1]), alpha.cpu().detach()[1])
plt.title('Alpha Values')
plt.xlabel('Index')
plt.ylabel('Alpha')
plt.show()

## II. Two operators

In [ ]:
# --- Hyperparameters and setup ---
num_steps = 1
epochs = 500

# Test data for extrapolation
x_test = inp[:, -1].to(device)
y_test = target.to(device)

training_horizon = 1
n_future_steps = 34

# --- Optimizer and Scheduler ---
alpha = torch.zeros(x_test.shape[0], num_operators).cuda().clone().detach().requires_grad_()
#alpha = torch.ones(x_test.shape[0], num_operators).cuda().clone().detach().requires_grad_()
theta_ = theta.clone().detach().requires_grad_()
lambd = 0.01  # 1e-3
optimizer = torch.optim.AdamW([
    {'params': [alpha], 'lr': 1},          # Learning rate for alpha
], weight_decay=0)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)
#num_training_frames = 3
noise_level = 0

print("Starting training...")

for epoch in tqdm(range(epochs), desc="Training"):

    if epoch %100==0 and epoch>0:
        scheduler.step()
    
    # Random time step for training
    t = random.randint(0, n_input_frames - training_horizon - 1)
    x_train = inp[:, t].to(device)
    y_train = inp[:, t+1:t+1+training_horizon].to(device)
    y_train = rearrange(y_train, "b t c h -> t b c h")

    # Random time step for old trajectory

    t = random.randint(0, n_input_frames+n_output_frames - training_horizon - 1)
    x_train_ = trajectory_train[:, t].to(device)
    x_train_ = x_train_ + torch.randn_like(x_train)*noise_level
    y_train_ = trajectory_train[:, t+1:t+1+training_horizon].to(device)
    y_train_ = rearrange(y_train_, "b t c h -> t b c h")

    operators = get_batched_operators(theta_, model, dim=1)

    operators1 = get_batched_operators(theta_[:num_operators//2], model, dim=1)
    operators2 = get_batched_operators(theta_[num_operators//2:], model, dim=1)

    pred = []
    current = x_train
    for _ in range(training_horizon):
        tmp = solve_mixture_ode(
            current, operators1, alpha[:, :num_operators//2], state_labels,
            integration_time=1, n_future_steps=1)
        current = solve_mixture_ode(
            tmp[-1], operators2, alpha[:, num_operators//2:], state_labels,
            integration_time=1, n_future_steps=1)
        pred.append(current)
        current = current[-1]

    pred = torch.cat(pred, axis=0)

    pred_ = solve_mixture_ode(
    x_train_, operators, alpha_train.cuda(), state_labels,
    integration_time=training_horizon, n_future_steps=training_horizon)
        
    # Calculate training loss
    rel_loss = relative_l2_error(pred, y_train)
    rel_loss_2 = relative_l2_error(pred_, y_train_)
    sparsity_loss = alpha.abs().sum(-1).mean()
    #loss = rel_loss + 0.01*rel_loss_2 + lambd*sparsity_loss
    loss = rel_loss + rel_loss_2 + lambd*sparsity_loss
    
    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    
    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Train Loss: {rel_loss.item():.6f} | "
        f"Train Loss 2: {rel_loss_2.item():.6f} | "
        f"Sparsity Loss: {sparsity_loss.item():.6f} | "
        #f"Test Error: {test_error:.6f}"
    )

print("Training finished.")

In [ ]:
alpha

In [ ]:
import torch

def sparsify_weights(weights, ratio_threshold=0.1):
    """
    Set weights to zero if they are less than ratio_threshold * max_weight
    
    Args:
        weights: tensor of weights
        ratio_threshold: threshold ratio (default 0.1 for 10%)
    
    Returns:
        sparsified weights
    """
    max_weight = torch.max(torch.abs(weights), dim=-1)[0]
    threshold = ratio_threshold * max_weight
    
    # Create mask for weights above threshold
    mask = torch.abs(weights) >= threshold
    
    # Apply mask
    sparsified_weights = weights * mask
    
    return sparsified_weights

# Your example
#weights = torch.tensor([[1.8809, 0.0901]])
#parsified = sparsify_weights(weights, ratio_threshold=0.1)

#print(f"Original: {weights}")
#print(f"Sparsified: {sparsified}")
#print(f"Threshold used: {0.1 * torch.max(torch.abs(weights)):.4f}")

In [ ]:
#alpha = weights.cuda()

In [ ]:
# --- Hyperparameters and setup ---
num_steps = 1
epochs = 600

# Test data for extrapolation
x_test = inp[:, -1].to(device)
y_test = target.to(device)

training_horizon = 1
n_future_steps = 34

# --- Optimizer and Scheduler ---
#alpha = torch.zeros(x_test.shape[0], num_operators).cuda().clone().detach().requires_grad_()
#alpha_ = 10*torch.ones(x_test.shape[0], num_operators).cuda().clone().detach().requires_grad_()
#alpha_ = torch.tensor([0, 0, 0, 10.]).unsqueeze(0).cuda().clone().detach().requires_grad_()
alpha_ = sparsify_weights(alpha).clone().detach().requires_grad_()
theta_ = theta.clone().detach().requires_grad_()
lambd = 1e-3 # 0 for 0.9x 0.9
optimizer = torch.optim.AdamW([
    {'params': [alpha_], 'lr': 1e-3},          # Learning rate for alpha
    {'params': [theta_], 'lr': 1e-4} # 1e-4 before
], weight_decay=0)

scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
#num_training_frames = 3
noise_level = 1e-3

print("Starting training...")

for epoch in tqdm(range(epochs), desc="Training"):

    if epoch % 100==0:
        alpha_.data = sparsify_weights(alpha_)
        if alpha_.grad is not None:
            alpha_.grad.zero_()

    if epoch > 400:
        training_horizon=5
        
    if epoch > 500:
        training_horizon=10
        
    # Random time step for training
    t = random.randint(0, n_input_frames - training_horizon - 1)
    x_train = inp[:, t].to(device)
    y_train = inp[:, t+1:t+1+training_horizon].to(device)
    y_train = rearrange(y_train, "b t c h -> t b c h")

    # Random time step for old trajectory

    t = random.randint(0, n_input_frames+n_output_frames - training_horizon - 1)
    x_train_ = trajectory_train[:, t].to(device)
    x_train_ = x_train_ + torch.randn_like(x_train)*noise_level
    y_train_ = trajectory_train[:, t+1:t+1+training_horizon].to(device)
    y_train_ = rearrange(y_train_, "b t c h -> t b c h")

    operators = get_batched_operators(theta_, model, dim=1)

    operators1 = get_batched_operators(theta_[:num_operators//2], model, dim=1)
    operators2 = get_batched_operators(theta_[num_operators//2:], model, dim=1)

    pred = []
    current = x_train
    for _ in range(training_horizon):
        tmp = solve_mixture_ode(
            current, operators1, alpha_[:, :num_operators//2], state_labels,
            integration_time=1, n_future_steps=1)
        current = solve_mixture_ode(
            tmp[-1], operators2, alpha_[:, num_operators//2:], state_labels,
            integration_time=1, n_future_steps=1)
        pred.append(current)
        current = current[-1]

    pred = torch.cat(pred, axis=0)

    pred_ = solve_mixture_ode(
    x_train_, operators, alpha_train.cuda(), state_labels,
    integration_time=training_horizon, n_future_steps=training_horizon)
        
    # Calculate training loss
    rel_loss = relative_l2_error(pred, y_train)
    rel_loss_2 = relative_l2_error(pred_, y_train_)
    sparsity_loss = alpha_.abs().sum(-1).mean()
    #loss = rel_loss + 0.01*rel_loss_2 + lambd*sparsity_loss
    loss = rel_loss + rel_loss_2 + lambd*sparsity_loss
    
    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    
    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Alpha [{alpha_}] | "
        f"Train Loss: {rel_loss.item():.6f} | "
        f"Train Loss 2: {rel_loss_2.item():.6f} | "
        f"Sparsity Loss: {sparsity_loss.item():.6f} | "
        #f"Test Error: {test_error:.6f}"
    )

print("Training finished.")

In [ ]:
alpha_ = sparsify_weights(alpha_)
pred=[]
with torch.no_grad():
    current = x_test
    for _ in range(n_output_frames):
        tmp = solve_mixture_ode(
            current, operators1, alpha_[:, :num_operators//2], state_labels,
            integration_time=1, n_future_steps=1)
        current = solve_mixture_ode(
            tmp[-1], operators2, alpha_[:, num_operators//2:], state_labels,
            integration_time=1, n_future_steps=1)
        pred.append(current)
        current = current[-1]

    pred = torch.cat(pred, axis=0)

In [ ]:
yhat = pred.detach().cpu()
y = rearrange(target.clone().cpu(), "b t c h -> t b c h")

In [ ]:
relative_l2_error(rearrange(yhat[:n_output_frames], "t b c h -> b t c h"), rearrange(y[:n_output_frames], "t b c h -> b t c h"))

In [ ]:
idx=0
for t in range(15):
    plt.plot(yhat[t, idx].squeeze())

In [ ]:
for t in range(10):
    #plt.plot(yhat[t, idx].squeeze())
    plt.plot(y[t, idx].squeeze())

In [ ]:
alpha_

In [ ]:
# --- Hyperparameters and setup ---
num_steps = 1
epochs = 1000
# Test data for extrapolation
x_test = inp[:, -1].to(device)
y_test = target.to(device)
training_horizon = 1
n_future_steps = 34

# --- Initialize parameters ---
# Initialize alpha to zeros as requested
alpha_ = torch.zeros(x_test.shape[0], num_operators).cuda().clone().detach().requires_grad_()
theta_ = theta.clone().detach().requires_grad_()

lambd = 1e-2 # 0 for 0.9x 0.9

# --- Create separate optimizers for alternating optimization ---
optimizer_alpha = torch.optim.AdamW([alpha_], lr=1e-1, weight_decay=0)
optimizer_theta = torch.optim.AdamW([theta_], lr=1e-4, weight_decay=0)

# --- Schedulers for both optimizers ---
scheduler_alpha = CosineAnnealingLR(optimizer_alpha, T_max=epochs)
scheduler_theta = CosineAnnealingLR(optimizer_theta, T_max=epochs)

noise_level = 1e-3
step_count = 0  # Track global step count for alternating

print("Starting alternating optimization training...")
for epoch in tqdm(range(epochs), desc="Training"):
    if epoch % 100 == 0:
        alpha_.data = sparsify_weights(alpha_)
        if alpha_.grad is not None:
            alpha_.grad.zero_()
    
    if epoch > 500:
        training_horizon = 2
        
    # Random time step for training
    t = random.randint(0, n_input_frames - training_horizon - 1)
    x_train = inp[:, t].to(device)
    y_train = inp[:, t+1:t+1+training_horizon].to(device)
    y_train = rearrange(y_train, "b t c h -> t b c h")
    
    # Random time step for old trajectory
    t = random.randint(0, n_input_frames+n_output_frames - training_horizon - 1)
    x_train_ = trajectory_train[:, t].to(device)
    x_train_ = x_train_ + torch.randn_like(x_train)*noise_level
    y_train_ = trajectory_train[:, t+1:t+1+training_horizon].to(device)
    y_train_ = rearrange(y_train_, "b t c h -> t b c h")
    
    operators = get_batched_operators(theta_, model, dim=1)
    operators1 = get_batched_operators(theta_[:num_operators//2], model, dim=1)
    operators2 = get_batched_operators(theta_[num_operators//2:], model, dim=1)
    
    pred = []
    current = x_train
    for _ in range(training_horizon):
        tmp = solve_mixture_ode(
            current, operators1, alpha_[:, :num_operators//2], state_labels,
            integration_time=1, n_future_steps=1)
        current = solve_mixture_ode(
            tmp[-1], operators2, alpha_[:, num_operators//2:], state_labels,
            integration_time=1, n_future_steps=1)
        pred.append(current)
        current = current[-1]
    pred = torch.cat(pred, axis=0)
    
    pred_ = solve_mixture_ode(
        x_train_, operators, alpha_train.cuda(), state_labels,
        integration_time=training_horizon, n_future_steps=training_horizon)
        
    # Calculate training loss
    rel_loss = relative_l2_error(pred, y_train)
    rel_loss_2 = relative_l2_error(pred_, y_train_)
    sparsity_loss = alpha_.abs().sum(-1).mean()
    loss = rel_loss + rel_loss_2 + lambd*sparsity_loss
    
    # --- Alternating optimization logic ---
    # Determine which parameters to optimize based on step count
    optimize_alpha = (step_count % 100) < 50  # First 25 steps: alpha, next 25: theta
    
    if optimize_alpha:
        # Optimize alpha only
        optimizer_alpha.zero_grad()
        loss.backward()
        # Zero out theta gradients to prevent updates
        if theta_.grad is not None:
            theta_.grad.zero_()
        optimizer_alpha.step()
        scheduler_alpha.step()
        opt_mode = "Alpha"
    else:
        # Optimize theta only
        optimizer_theta.zero_grad()
        loss.backward()
        # Zero out alpha gradients to prevent updates
        if alpha_.grad is not None:
            alpha_.grad.zero_()
        optimizer_theta.step()
        scheduler_theta.step()
        opt_mode = "Theta"
    
    step_count += 1
    
    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Mode: {opt_mode} | "
        f"Alpha [{alpha_}] | "
        f"Train Loss: {rel_loss.item():.6f} | "
        f"Train Loss 2: {rel_loss_2.item():.6f} | "
        f"Sparsity Loss: {sparsity_loss.item():.6f}"
    )

print("Training finished.")

In [ ]:
alpha_ = sparsify_weights(alpha_)
pred=[]
with torch.no_grad():
    current = x_test
    for _ in range(n_output_frames):
        tmp = solve_mixture_ode(
            current, operators1, alpha_[:, :num_operators//2], state_labels,
            integration_time=1, n_future_steps=1)
        current = solve_mixture_ode(
            tmp[-1], operators2, alpha_[:, num_operators//2:], state_labels,
            integration_time=1, n_future_steps=1)
        pred.append(current)
        current = current[-1]

    pred = torch.cat(pred, axis=0)

In [ ]:
yhat = pred.detach().cpu()
y = rearrange(target.clone().cpu(), "b t c h -> t b c h")

In [ ]:
relative_l2_error(rearrange(yhat[:n_output_frames], "t b c h -> b t c h"), rearrange(y[:n_output_frames], "t b c h -> b t c h"))

In [ ]:
# --- Hyperparameters and setup ---
num_steps = 1
epochs = 1000

# Test data for extrapolation
x_test = inp[:, -1].to(device)
y_test = target.to(device)

# --- Optimizer and Scheduler ---
theta_latent1 = theta_latent1_.clone().detach().requires_grad_()
theta_latent2 = theta_latent2_.clone().detach().requires_grad_()
optimizer = torch.optim.AdamW([theta_latent1, theta_latent2], lr=1e-4, weight_decay=0)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

num_training_frames = 1

print("Starting training...")

for epoch in tqdm(range(epochs), desc="Training"):
    model.train()
    
    # Random time step for training
    t = random.randint(0, n_input_frames - num_training_frames - 1)
    x_train = inp[:, t]
    y_train = inp[:, t + 1]

    theta1 = model.decode_theta(theta_latent1, dim)
    theta2 = model.decode_theta(theta_latent2, dim)

    pred = []
    current_state = x_train
    
    for _ in range(num_training_frames):
        for _ in range(num_steps):
            # Apply first operator
            pred_int, _ = model.solve_ode(
                current_state, theta1, state_labels, dim, 
                n_future_steps=1, integration_time=1/num_steps, 
                dt=1/num_steps, predict_normed=False, metadata={}
            )
            # Apply second operator
            pred_step, _ = model.solve_ode(
                pred_int[:, -1], theta2, state_labels, dim, 
                n_future_steps=1, integration_time=1/num_steps, 
                dt=1/num_steps, predict_normed=False, metadata={}
            )
            current_state = pred_step[:, -1]
        
        pred.append(current_state)
        x_train = current_state
        
    pred = torch.cat(pred, 1)
    
    # Calculate training loss
    loss = relative_l2_error(pred, y_train)
    
    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    # Evaluation
    if (epoch + 1) % 10 == 0 or epoch == 0:
        with torch.no_grad():
            model.eval()
            
            theta1 = model.decode_theta(theta_latent1, dim)
            theta2 = model.decode_theta(theta_latent2, dim)
            
            pred = []
            current_state = x_test.clone()
            
            for _ in range(n_output_frames):
                for _ in range(num_steps):
                    # Apply first operator
                    pred_int, _ = model.solve_ode(
                        current_state, theta1, state_labels, dim, 
                        n_future_steps=1, integration_time=1/num_steps, 
                        dt=1/num_steps, predict_normed=False, metadata={}
                    )
                    # Apply second operator
                    pred_step, _ = model.solve_ode(
                        pred_int[:, -1], theta2, state_labels, dim, 
                        n_future_steps=1, integration_time=1/num_steps, 
                        dt=1/num_steps, predict_normed=False, metadata={}
                    )
                    current_state = pred_step[:, -1]
                
                pred.append(current_state)
                
            pred = torch.cat(pred, 1)
            test_error = relative_l2_error(pred, y_test).item()
        
        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {loss.item():.6f} | "
            f"Test Error: {test_error:.6f}"
        )

print("Training finished.")

In [ ]:
# alternate

# II. manual composition: i.e. what we want to reach by optimization

In [ ]:
class TemporalDatasetFixedCI(torch.utils.data.Dataset):
    def __init__(self, n_batches, batch_size, sub_x, sub_t, split="train", input_frames=16, output_frames=2,
                 L=16.0, nx=256, nt=100, T=10.0,
                 v_range=[0.01, 0.025, 0.05, 0.1, 0.5, 1.0], D_range=[0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0],
                 fractal_degree=8, fractal_power=2, seed=None):
        self.n_batches = n_batches
        self.batch_size = batch_size
        self.sub_x = sub_x
        self.sub_t = sub_t
        self.split = split
        self.input_frames = input_frames
        self.output_frames = output_frames
        self.L = L
        self.nx = nx
        self.nt = nt
        self.T = T
        self.v_range = v_range
        self.D_range = D_range
        self.fractal_degree = fractal_degree
        self.fractal_power = fractal_power
        self.seed = seed
        self.rng = np.random.default_rng(seed)
        self.u0 = []
        for _ in range(self.batch_size):
            fractaloid = Fractaloid(
                degree=self.fractal_degree,
                power=self.fractal_power,
                size=self.nx,
                patch_size=self.nx
            )
            u0 = fractaloid.generate(batch_size=1, seed=None).squeeze(0).numpy()
            u0 = (u0 - u0.mean()) / (u0.std() + 1e-8)
            self.u0.append(torch.from_numpy(u0))
            
        self.u0 = torch.stack(self.u0)

    def __len__(self):
        return len(self.u0)

    def __getitem__(self, idx):
    
        input_frames = self.input_frames
        batch_inputs = []
        batch_targets = []
        batch_v = []
        batch_d = []
        batch_init = []
        for v in self.v_range:
            for d in self.D_range:
                u0 = self.u0[idx]
                u_xt, x, t = advection_diffusion_analytical(
                    u0, L=self.L, v=v, D=d, nt=self.nt, T=self.T
                )
                u_xt = u_xt[::self.sub_t, ::self.sub_x]
                input = u_xt[:input_frames].copy()
                target = u_xt[input_frames: input_frames + self.output_frames].copy()
                
                batch_inputs.append(torch.from_numpy(input).unsqueeze(-2).float())
                batch_targets.append(torch.from_numpy(target).unsqueeze(-2).float())
                batch_v.append(v)
                batch_d.append(d)
                batch_init.append(u0)
                
        batch = {
            'input': torch.stack(batch_inputs),
            'target': torch.stack(batch_targets),
            'velocities': batch_v,
            'diffusivities': batch_d,
            'initial_conditions': torch.stack(batch_init)
        }
        return batch

In [ ]:
n_batches = 1  # or set as needed for your epoch size
batch_size = 128
split="test"
n_input_frames=16
n_output_frames=34
model.eval()
num_steps=1

#composition_type="composition"
#composition_type="sum"

#advection_speeds = [0.59, 0., 0.59]

advection_speeds = [1.0, 0., 1.0]
viscosities = [0., 1.0, 1.0]

#advection_speeds = [0.9, 0.9, 1.8]
#viscosities = [0., 0, 0.]

#advection_speeds = [0., 0., 0.]
#viscosities = [0.95, 0.95, 1.9]


all_velocities = []
all_diffusivities = []
all_input = []
all_target = []
all_theta_latent = []
all_theta = []

train_ds = TemporalDatasetFixedCI(
        n_batches=n_batches,
        batch_size=batch_size,
        sub_x=sub_x,
        sub_t=sub_t,
        split=split,
        input_frames=n_input_frames,
        output_frames=n_output_frames,
        L=16.0,
        nx=256,
        nt=100,
        T=10.0,
        fractal_power=3.0,
        fractal_degree=256, # nx
        v_range=[0],#(0.01, 1.0),
        D_range=[0], #(0.001, 1.0),
    )


for advection_speed, viscosity in zip(advection_speeds, viscosities):

    train_ds.v_range=[advection_speed]
    train_ds.D_range=[viscosity]
    train_loader = DataLoader(train_ds, batch_size=batch_size, num_workers=1, prefetch_factor=1, pin_memory=True, shuffle=False)
    
    for batch in tqdm(train_loader):
        inp, target = batch["input"], batch["target"]
        inp = inp.squeeze(1)
        target = target.squeeze(1)
        print('inp', inp.shape, target.shape)
        all_velocities += batch["velocities"]
        all_diffusivities += batch["diffusivities"]
        
    inp = inp.to(device)
    target = target.to(device)
    state_labels = torch.tensor([0], device=inp.device)
    
    x_shape = inp.shape
    B, T, C = x_shape[:3]
    spatial = x_shape[3:]
    dim = len(spatial)
    
    n_sample = inp.shape[0]
    #pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)
    
    #predict
    with torch.no_grad():
        # encode into 2 dimensional
        theta_latent, metadata= model.encode_theta_latent(inp, state_labels)
        # decode into 100k parameters
        theta = model.decode_theta(theta_latent, dim)
        #pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=n_output_frames, predict_normed=False, metadata=metadata)
        pred = []
        current_state = inp[:, -1]
        for _ in range(n_output_frames):
            for _ in range(num_steps):
                pred_step, _ = model.solve_ode(
                    current_state, theta, state_labels, dim, 
                    n_future_steps=1, integration_time=1/num_steps, 
                    dt=1/num_steps, predict_normed=False, metadata={}
                )
                current_state = pred_step[:,-1]
            pred.append(current_state)
        pred = torch.cat(pred, 1)
    rollout_error = relative_l2_error(pred, target[:, :n_output_frames]).item()
    
    print(f"Initial error", rollout_error)

    all_theta_latent.append(theta_latent)
    all_theta.append(theta)
    all_input.append(inp)
    all_target.append(target)


all_theta_latent = torch.stack(all_theta_latent)
all_theta = torch.stack(all_theta)
all_input = torch.stack(all_input)
all_target = torch.stack(all_target)

In [ ]:
### setup the data 
theta1 = all_theta[0]
theta2 = all_theta[1]

x_test = all_input[2]
y_test = all_target[2]
state_labels = torch.tensor([0], device=x_test.device)

In [ ]:
#model.max_steps=5
theta_latent.shape

In [ ]:
pred_test = []
x_test_ = x_test[:, -1].clone()
n_output_frames=34
composition_type="composition"
num_steps=1
solver="rk4"
n_future_steps=1

if composition_type=="sum":
    with torch.no_grad():
        pred_test, _ = model.solve_ode_with_2_operators(x_test_, theta1, theta2, state_labels, dim, n_future_steps=n_output_frames, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
    )
else:
    with torch.no_grad():
        current_state = x_test_.clone()
        pred=[]
        for _ in range(n_output_frames):
            pred_int, _ = model.solve_ode(
            current_state, theta2, state_labels, dim, integration_time=1,
            n_future_steps=1, predict_normed=False, metadata={}, solver=solver)
            
            # Apply second operator
            pred_step, _ = model.solve_ode(
                pred_int[:, -1], theta1, state_labels, dim, integration_time=1, 
                n_future_steps=1, predict_normed=False, metadata={}, solver=solver
            )
            current_state = pred_step[:, -1]            
            pred.append(current_state)
            
        pred = torch.cat(pred, 1)
# Calculate the test error.
test_error = relative_l2_error(pred, y_test[:, :n_output_frames], aggregate=None)
print(test_error.mean())

In [ ]:
relative_l2_error(pred[:, :10], y_test[:, :10])

In [ ]:
#print('test_error', test_error)
idx = 0
for t in range(10):
    plt.plot(pred.detach().cpu().numpy()[idx, t].squeeze(), label=t)
plt.legend()

In [ ]:
for t in range(10):
    plt.plot(y_test.detach().cpu().numpy()[idx, t].squeeze(), label=t)
plt.legend()

In [ ]:
# --- Hyperparameters and setup ---
epochs = 100
n_output_frames = 34
training_horizon = 5
solver="rk4"

theta1_time = []
theta2_time = []

# --- Optimizer and Scheduler ---
theta_latent1 = theta_latent1_.clone().detach().requires_grad_() #(torch.randn_like(theta_latent.detach())).requires_grad_()
theta_latent2 = theta_latent2_.clone().detach().requires_grad_() #(torch.randn_like(theta_latent.detach())).requires_grad_()

optimizer = torch.optim.AdamW([theta_latent1, theta_latent2], lr=1, weight_decay=0, betas=[0.5,0.5])
#optimizer = torch.optim.SGD([theta_latent1, theta_latent2], lr=10, weight_decay=0)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

print("Starting training...")

for epoch in tqdm(range(epochs), desc="Training"):
    model.train()
    
    # Random time step for training
    t = random.randint(0, n_input_frames - training_horizon - 1)

    # Decode latent parameters
    theta1 = model.decode_theta(theta_latent1, dim)
    theta2 = model.decode_theta(theta_latent2, dim)

    # Training prediction using composition
    pred = []
    current_state = x_test[:, t]
    
    for _ in range(training_horizon):
        # Apply first operator
        pred_int, _ = model.solve_ode(
            current_state, theta1, state_labels, dim, integration_time=1,
            n_future_steps=1, predict_normed=False, metadata={}, solver=solver
        )
        # Apply second operator
        pred_step, _ = model.solve_ode(
            pred_int[:, -1], theta2, state_labels, dim, integration_time=1, 
            n_future_steps=1, predict_normed=False, metadata={}, solver=solver
        )
        current_state = pred_step[:, -1]
        pred.append(current_state)

    pred = torch.cat(pred, 1)
    
    # Calculate training loss
    loss = relative_l2_error(pred, x_test[:, t+1:t+training_horizon+1])
    
    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Store theta evolution
    theta1_time.append(theta_latent1.detach().cpu())
    theta2_time.append(theta_latent2.detach().cpu())
    
    scheduler.step()

    # Evaluation
    if (epoch + 1) % 50 == 0 or epoch == 0:
        with torch.no_grad():
            model.eval()
            
            # Decode latent parameters
            theta1 = model.decode_theta(theta_latent1, dim)
            theta2 = model.decode_theta(theta_latent2, dim)

            # Test prediction using composition
            pred_test = []
            current_state = x_test[:, -1]
            
            for _ in range(n_output_frames):
                # Apply first operator
                pred_int, _ = model.solve_ode(
                    current_state, theta1, state_labels, dim, integration_time=1,
                    n_future_steps=1, predict_normed=False, metadata={}
                )
                # Apply second operator
                pred_step, _ = model.solve_ode(
                    pred_int[:, -1], theta2, state_labels, dim, integration_time=1,
                    n_future_steps=1, predict_normed=False, metadata={}
                )
                current_state = pred_step[:, -1]
                pred_test.append(pred_step)

            pred_test = torch.cat(pred_test, 1)
            test_error = relative_l2_error(pred_test, y_test).item()
    
        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {loss.item():.6f} | "
            f"Test Error: {test_error:.6f}"
        )

print("Training finished.")

In [ ]:
theta1_time=torch.stack(theta1_time)
theta2_time=torch.stack(theta2_time)

In [ ]:

plt.plot(theta1_time[:, idx, 0], theta1_time[:, idx, 1], linestyle="--",c="gray", zorder=1)
plt.scatter(theta1_time[:, idx, 0], theta1_time[:, idx, 1], c=[t for t in range(theta1_time.shape[0])], zorder=2)
plt.scatter(all_theta_latent[0, idx, 0].cpu().detach(), all_theta_latent[0, idx, 1].cpu().detach(), c="red", label="theta1", zorder=3)
plt.scatter(all_theta_latent[1, idx, 0].cpu().detach(), all_theta_latent[1, idx, 1].cpu().detach(), c="orange", label="theta2", zorder=4)
plt.legend()
plt.colorbar()

In [ ]:

plt.plot(theta2_time[:, idx, 0], theta2_time[:, idx, 1], linestyle="--",c="gray", zorder=1)
plt.scatter(theta2_time[:, idx, 0], theta2_time[:, idx, 1], c=[t for t in range(theta1_time.shape[0])], zorder=2)
plt.scatter(all_theta_latent[0, idx, 0].cpu().detach(), all_theta_latent[0, idx, 1].cpu().detach(), c="red", label="theta1", zorder=3)
plt.scatter(all_theta_latent[1, idx, 0].cpu().detach(), all_theta_latent[1, idx, 1].cpu().detach(), c="orange", label="theta2", zorder=4)
plt.legend()
plt.colorbar()

In [ ]:
#print('test_error', test_error)
idx = 0
for t in range(5):
    plt.plot(pred_test.detach().cpu().numpy()[idx, t].squeeze(), label=t)
plt.legend()

In [ ]:
# --- Hyperparameters and setup ---
epochs = 500
n_output_frames = 34
training_horizon = 1
solver="rk4"

theta1_time = []
theta2_time = []

# --- Optimizer and Scheduler ---
theta1 = theta1.clone().detach().requires_grad_() #(torch.randn_like(theta_latent.detach())).requires_grad_()
theta2 = theta2.clone().detach().requires_grad_() #(torch.randn_like(theta_latent.detach())).requires_grad_()

optimizer = torch.optim.AdamW([theta1, theta2], lr=1e-3, weight_decay=0, betas=[0.9,0.99])
#optimizer = torch.optim.SGD([theta_latent1, theta_latent2], lr=10, weight_decay=0)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

print("Starting training...")

for epoch in tqdm(range(epochs), desc="Training"):
    model.train()
    
    # Random time step for training
    t = random.randint(0, n_input_frames - training_horizon - 1)

    # Decode latent parameters
    #theta1 = model.decode_theta(theta_latent1, dim)
    #theta2 = model.decode_theta(theta_latent2, dim)

    # Training prediction using composition
    pred = []
    current_state = x_test[:, t]
    
    for _ in range(training_horizon):
        # Apply first operator
        pred_int, _ = model.solve_ode(
            current_state, theta1, state_labels, dim, integration_time=1,
            n_future_steps=1, predict_normed=False, metadata={}, solver=solver
        )
        # Apply second operator
        pred_step, _ = model.solve_ode(
            pred_int[:, -1], theta2, state_labels, dim, integration_time=1, 
            n_future_steps=1, predict_normed=False, metadata={}, solver=solver
        )
        current_state = pred_step[:, -1]
        pred.append(current_state)

    pred = torch.cat(pred, 1)
    
    # Calculate training loss
    loss = relative_l2_error(pred, x_test[:, t+1:t+training_horizon+1])
    
    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Store theta evolution
    theta1_time.append(theta_latent1.detach().cpu())
    theta2_time.append(theta_latent2.detach().cpu())
    
    scheduler.step()

    # Evaluation
    if (epoch + 1) % 50 == 0 or epoch == 0:
        with torch.no_grad():
            model.eval()
            
            # Decode latent parameters
            #theta1 = model.decode_theta(theta_latent1, dim)
            #theta2 = model.decode_theta(theta_latent2, dim)

            # Test prediction using composition
            pred_test = []
            current_state = x_test[:, -1]
            
            for _ in range(n_output_frames):
                # Apply first operator
                pred_int, _ = model.solve_ode(
                    current_state, theta1, state_labels, dim, integration_time=1,
                    n_future_steps=1, predict_normed=False, metadata={}
                )
                # Apply second operator
                pred_step, _ = model.solve_ode(
                    pred_int[:, -1], theta2, state_labels, dim, integration_time=1,
                    n_future_steps=1, predict_normed=False, metadata={}
                )
                current_state = pred_step[:, -1]
                pred_test.append(pred_step)

            pred_test = torch.cat(pred_test, 1)
            test_error = relative_l2_error(pred_test, y_test).item()
    
        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {loss.item():.6f} | "
            f"Test Error: {test_error:.6f}"
        )

print("Training finished.")

In [ ]:
# --- Hyperparameters and setup ---
epochs = 500
n_output_frames = 34
training_horizon = 1
solver = "rk4"
theta_time = []

# --- Optimizer and Scheduler ---
theta_latent = theta_latent1_.clone().detach().requires_grad_()  # Use only first operator
optimizer = torch.optim.AdamW([theta_latent], lr=1e-1, weight_decay=0, betas=[0.5, 0.9])
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

print("Starting training...")
for epoch in tqdm(range(epochs), desc="Training"):
    model.train()
    
    # Random time step for training
    t = random.randint(0, n_input_frames - training_horizon - 1)
    
    # Decode latent parameters
    theta = model.decode_theta(theta_latent, dim)
    
    # Training prediction using single operator
    pred = []
    current_state = x_test[:, t]
    
    for _ in range(training_horizon):
        # Apply single operator
        pred_step, _ = model.solve_ode(
            current_state, theta, state_labels, dim, integration_time=1,
            n_future_steps=1, predict_normed=False, metadata={}, solver=solver
        )
        current_state = pred_step[:, -1]
        pred.append(current_state)
    
    pred = torch.cat(pred, 1)
    
    # Calculate training loss
    loss = relative_l2_error(pred, x_test[:, t+1:t+training_horizon+1])
    
    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Store theta evolution
    theta_time.append(theta_latent.detach().cpu())
    
    scheduler.step()
    
    # Evaluation
    if (epoch + 1) % 50 == 0 or epoch == 0:
        with torch.no_grad():
            model.eval()
            
            # Decode latent parameters
            theta = model.decode_theta(theta_latent, dim)
            
            # Test prediction using single operator
            pred_test = []
            current_state = x_test[:, -1]
            
            for _ in range(n_output_frames):
                # Apply single operator
                pred_step, _ = model.solve_ode(
                    current_state, theta, state_labels, dim, integration_time=1,
                    n_future_steps=1, predict_normed=False, metadata={}
                )
                current_state = pred_step[:, -1]
                pred_test.append(pred_step)
            
            pred_test = torch.cat(pred_test, 1)
            test_error = relative_l2_error(pred_test, y_test).item()
    
        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {loss.item():.6f} | "
            f"Test Error: {test_error:.6f}"
        )

print("Training finished.")

In [ ]:
#print('test_error', test_error)
idx = 0
for t in range(10):
    plt.plot(pred_test.detach().cpu().numpy()[idx, t].squeeze(), label=t)
plt.legend()

In [ ]:
theta_time = np.array(theta_time)
plt.plot(theta_time[:, idx, 0], theta_time[:, idx, 1], linestyle="--",c="gray", zorder=1)
plt.scatter(theta_time[:, idx, 0], theta_time[:, idx, 1], c=[t for t in range(theta_time.shape[0])], zorder=2)
plt.scatter(all_theta_latent[0, idx, 0].cpu().detach(), all_theta_latent[0, idx, 1].cpu().detach(), c="red", label="theta1", zorder=3)
plt.scatter(all_theta_latent[1, idx, 0].cpu().detach(), all_theta_latent[1, idx, 1].cpu().detach(), c="orange", label="theta2", zorder=4)
plt.legend()
plt.colorbar()

In [ ]:
np.array(theta_time).shape

In [ ]:
# --- Hyperparameters and setup ---
epochs = 500
n_output_frames = 34
training_horizon = 1
solver="rk4"

theta1_time = []
theta2_time = []

# --- Optimizer and Scheduler ---
theta_latent1 = theta_latent.clone().detach().requires_grad_() #(torch.randn_like(theta_latent.detach())).requires_grad_()
theta_latent2 = theta_latent2_.clone().detach().requires_grad_() #(torch.randn_like(theta_latent.detach())).requires_grad_()

optimizer = torch.optim.AdamW([theta_latent1, theta_latent2], lr=1e-1, weight_decay=0, betas=[0.5,0.9])
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

print("Starting training...")

for epoch in tqdm(range(epochs), desc="Training"):
    model.train()
    
    # Random time step for training
    t = random.randint(0, n_input_frames - training_horizon - 1)

    # Decode latent parameters
    theta1 = model.decode_theta(theta_latent1, dim)
    theta2 = model.decode_theta(theta_latent2, dim)

    # Training prediction using composition
    pred = []
    current_state = x_test[:, t]
    
    for _ in range(training_horizon):
        # Apply first operator
        pred_int, _ = model.solve_ode(
            current_state, theta1, state_labels, dim, integration_time=1,
            n_future_steps=1, predict_normed=False, metadata={}, solver=solver
        )
        # Apply second operator
        pred_step, _ = model.solve_ode(
            pred_int[:, -1], theta2, state_labels, dim, integration_time=1, 
            n_future_steps=1, predict_normed=False, metadata={}, solver=solver
        )
        current_state = pred_step[:, -1]
        pred.append(current_state)

    pred = torch.cat(pred, 1)
    
    # Calculate training loss
    loss = relative_l2_error(pred, x_test[:, t+1:t+training_horizon+1])
    
    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Store theta evolution
    theta1_time.append(theta_latent1.detach().cpu())
    theta2_time.append(theta_latent2.detach().cpu())
    
    scheduler.step()

    # Evaluation
    if (epoch + 1) % 50 == 0 or epoch == 0:
        with torch.no_grad():
            model.eval()
            
            # Decode latent parameters
            theta1 = model.decode_theta(theta_latent1, dim)
            theta2 = model.decode_theta(theta_latent2, dim)

            # Test prediction using composition
            pred_test = []
            current_state = x_test[:, -1]
            
            for _ in range(n_output_frames):
                # Apply first operator
                pred_int, _ = model.solve_ode(
                    current_state, theta1, state_labels, dim, integration_time=1,
                    n_future_steps=1, predict_normed=False, metadata={}
                )
                # Apply second operator
                pred_step, _ = model.solve_ode(
                    pred_int[:, -1], theta2, state_labels, dim, integration_time=1,
                    n_future_steps=1, predict_normed=False, metadata={}
                )
                current_state = pred_step[:, -1]
                pred_test.append(pred_step)

            pred_test = torch.cat(pred_test, 1)
            test_error = relative_l2_error(pred_test, y_test).item()
    
        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {loss.item():.6f} | "
            f"Test Error: {test_error:.6f}"
        )

print("Training finished.")

In [ ]:
model = model.cuda()

In [ ]:
# --- Hyperparameters and setup ---
# Use a more descriptive variable name for num_steps.
idx_=0
epochs = 500
n_output_frames=34

theta1_time = []
theta2_time = []

training_horizon=1

# Training data for interpolation.
#x_train = rearrange(inp[:, :-1].clone(), "b t c h -> (b t) 1 c h").to(device)
#y_train = rearrange(inp[:, 1:].clone(), "b t c h -> (b t) 1 c h").to(device)

# Test data for extrapolation.
#x_test = inp[:, -1].to(device)
#y_test = target.to(device)

# --- Optimizer and Scheduler ---
# Use standard PyTorch classes for clarity.
#theta1 = theta1.detach().clone().requires_grad_()
#theta2 = theta2.detach().clone().requires_grad_()

#theta_latent1 = (theta_latent.detach()+0.1*torch.randn_like(theta_latent.detach())).requires_grad_()
#theta_latent2 = (theta_latent.detach()+0.1*torch.randn_like(theta_latent.detach())).requires_grad_()
theta_latent1 = (torch.zeros_like(theta_latent.detach())).requires_grad_()
theta_latent2 = (torch.zeros_like(theta_latent.detach())).requires_grad_()

optimizer = torch.optim.AdamW([theta_latent1] + [theta_latent2], lr=1e-1)

# Use CosineAnnealingLR for a standard cosine decay schedule.
# This is a common and effective choice.
scheduler = CosineAnnealingLR(optimizer, T_max=epochs) 

print("Starting training...")
# Wrap the range in tqdm to get a progress bar.
for epoch in tqdm(range(epochs), desc="Training"):
    # --- Interpolation Step (Training) ---
    # Set the model to training mode (if applicable).
    # Some models might have different modes for training and inference.
    model.train()
    
    # Initialize metadata for the solve_ode call.
    # It's better to initialize it here if it's used within the loop.

    if epoch < 400:
        training_horizon=1
    else:
        training_horizon=10
    metadata = {} 

    t = random.randint(0, n_input_frames-training_horizon-1)

    theta1 = model.decode_theta(theta_latent1, dim)
    theta2 = model.decode_theta(theta_latent2, dim)

    if composition_type=="sum":
        pred, _ = model.solve_ode_with_2_operators(x_test[:, t], theta1, theta2, state_labels, dim, n_future_steps=training_horizon, predict_normed=False, metadata=metadata # Re-initialize metadata for the test run
    )
    else:
        pred=[]
        x_test_ = x_test[:, t].clone()
        for _ in range(training_horizon):
            pred_int, _ = model.solve_ode(x_test_, theta1, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)
            pred_, _ = model.solve_ode(pred_int[:, -1], theta2, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)
            pred.append(pred_[:, -1])
            x_test_ = pred_[:, -1]
        pred = torch.cat(pred, 1)
    #x_train = inp[:, t]
    #y_train = inp[:, t+1]
    
    # Run the model to get the prediction.
    
    
    # Calculate the training loss.
    loss = relative_l2_error(pred, x_test[:, t+1:t+training_horizon+1])# + 0.001*torch.abs(F.cosine_similarity(theta_latent1, theta_latent2, dim=1)).mean()
    
    # --- Backpropagation ---
    # A standard training step.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    theta1_time.append(theta_latent1.detach().cpu())
    theta2_time.append(theta_latent2.detach().cpu())
    
    # Update the learning rate.
    scheduler.step()

    # --- Evaluation Step (Extrapolation) ---
    # It's a good practice to evaluate the model without gradients.
        
    # --- Print progress ---
    # Print the losses in a clear, formatted way.
    # You can print less frequently to avoid excessive output.
    if (epoch + 1) % 50 == 0 or epoch == 0:
        with torch.no_grad():
            # Set the model to evaluation mode.
            # This is important for layers like BatchNorm or Dropout.
            model.eval()
            theta1 = model.decode_theta(theta_latent1, dim)
            theta2 = model.decode_theta(theta_latent2, dim)

            pred_test = []
            x_test_ = x_test[:, -1].clone()
            for _ in range(n_output_frames):
                if composition_type=="sum":
                    pred, _ = model.solve_ode_with_2_operators(x_test_, theta1, theta2, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)

                else:
                    pred_int, _ = model.solve_ode(x_test_, theta1, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)
                    pred, _ = model.solve_ode(pred_int[:,-1], theta2, state_labels, dim, n_future_steps=1, predict_normed=False, metadata=metadata)
                pred_test.append(pred)
                x_test_ = pred[:, -1]

            pred_test = torch.cat(pred_test, 1)
            # Calculate the test error.
            test_error = relative_l2_error(pred_test, y_test).item()
    
            
        print(
            f"Epoch [{epoch+1}/{epochs}] |",
            f"Interpolation next time-step (Loss): {loss.item():.6f} | "
            f"Extrapolation next time-step (Error): {test_error:.6f}"
        )

print("Training finished.")

In [ ]:
# --- Hyperparameters and setup ---
# Use a more descriptive variable name for num_steps.
idx_=0
epochs = 100

theta1_ = theta1.clone().requires_grad_()
theta2_ = theta2.clone().requires_grad_()
training_horizon=5
num_output_frames=34

optimizer = torch.optim.AdamW([theta1_] + [theta2_], lr=1e-4, weight_decay=1e-6)

# Use CosineAnnealingLR for a standard cosine decay schedule.
# This is a common and effective choice.
scheduler = CosineAnnealingLR(optimizer, T_max=epochs) 

print("Starting training...")
# Wrap the range in tqdm to get a progress bar.
for epoch in tqdm(range(epochs), desc="Training"):
    # --- Interpolation Step (Training) ---
    # Set the model to training mode (if applicable).
    # Some models might have different modes for training and inference.
    model.train()
    
    # Initialize metadata for the solve_ode call.
    # It's better to initialize it here if it's used within the loop.
    metadata = {} 

    t = random.randint(0, n_input_frames-1-training_horizon)

    pred = []
    current_state = x_test.clone()[:, t]
    for _ in range(training_horizon):
        # Apply first operator
        pred_int, _ = model.solve_ode(
            current_state, theta1_, state_labels, dim, integration_time=1,
            n_future_steps=1, predict_normed=False, metadata={}, solver=solver
        )
        # Apply second operator
        pred_step, _ = model.solve_ode(
            pred_int[:, -1], theta2_, state_labels, dim, integration_time=1, 
            n_future_steps=1, predict_normed=False, metadata={}, solver=solver
        )
        current_state = pred_step[:, -1]
        pred.append(current_state)

    pred = torch.cat(pred, 1)
    
    # Calculate the training loss.
    loss = relative_l2_error(pred, x_test[:, t+1:t+training_horizon+1])# + 0.001*torch.abs(F.cosine_similarity(theta_latent1, theta_latent2, dim=1)).mean()
    
    # --- Backpropagation ---
    # A standard training step.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Update the learning rate.
    scheduler.step()

    # --- Evaluation Step (Extrapolation) ---
    # It's a good practice to evaluate the model without gradients.
        
    # --- Print progress ---
    # Print the losses in a clear, formatted way.
    # You can print less frequently to avoid excessive output.
    if (epoch + 1) % 10 == 0 or epoch == 0:
        with torch.no_grad():
            # Set the model to evaluation mode.
            # This is important for layers like BatchNorm or Dropout.
            model.eval()

            pred_test = []
            x_test_ = x_test[:, -1].clone()
            for _ in range(num_output_frames):
                # Apply first operator
                pred_int, _ = model.solve_ode(
                    x_test_, theta1_, state_labels, dim, integration_time=1,
                    n_future_steps=1, predict_normed=False, metadata={}, solver=solver
                )
                # Apply second operator
                pred_step, _ = model.solve_ode(
                    pred_int[:, -1], theta2_, state_labels, dim, integration_time=1, 
                    n_future_steps=1, predict_normed=False, metadata={}, solver=solver
                )
                x_test_ = pred_step[:, -1]
                pred_test.append(x_test_)
        
            pred_test = torch.cat(pred_test, 1)
            test_error = relative_l2_error(pred_test, y_test).item()
    
            
        print(
            f"Epoch [{epoch+1}/{epochs}] |",
            f"Interpolation next time-step (Loss): {loss.item():.6f} | "
            f"Extrapolation next time-step (Error): {test_error:.6f}"
        )

print("Training finished.")